# 12 — Bản đồ ngành ICB và độ rộng thị trường

VNINDEX tăng 1% không cho biết thị trường đang mạnh hay yếu. Nó có thể là ba mã
vốn hoá lớn kéo lên trong khi 300 mã còn lại đỏ. Notebook này dựng hai thứ trả
lời đúng câu hỏi đó:

1. **Bản đồ ngành** — 19 ngành ICB cấp 2, ngành nào dẫn dắt và từ lúc nào
2. **Độ rộng thị trường (breadth)** — bao nhiêu phần trăm số mã thực sự tăng

Cộng thêm cách cây ICB được tổ chức, vì mọi notebook sau đều dùng nó để nhóm.

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, heatmap, hom_nay, lui_ngay

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · Cây ICB có hai tầng

FinLens phơi ra **cấp 2** (19 ngành, để nhìn tổng thể) và **cấp 4** (106 ngành,
để so peer). Cấp 4 là tầng bạn dùng khi so định giá: "Ngân hàng" cấp 2 gộp cả
ngân hàng quốc doanh lẫn ngân hàng tư nhân nhỏ, còn cấp 4 tách chúng ra.

In [2]:
cap2 = client.meta.sectors(level=2)
cap4 = client.meta.sectors(level=4)

print(f"cấp 2: {len(cap2)} ngành · cấp 4: {len(cap4)} ngành")
cap2[["icb", "name_vi"]].head(19)

cấp 2: 19 ngành · cấp 4: 106 ngành


,icb,name_vi
0,0500,Dầu khí
1,1300,Hóa chất
2,1700,Tài nguyên Cơ bản
3,2300,Xây dựng và Vật liệu
4,2700,Hàng & Dịch vụ Công nghiệp
5,3300,Ô tô và phụ tùng
6,3500,Thực phẩm và đồ uống
7,3700,Hàng cá nhân & Gia dụng
8,4500,Y tế
9,5300,Bán lẻ


Mỗi mã mang cả hai cấp trong `meta.symbols()`, nên bạn không phải join gì cả:

In [3]:
ma_hose = client.meta.symbols(exchange="HOSE", kind="stock")
ma_hose[["symbol", "short_name", "icb_level2", "icb_name2", "icb_level4", "icb_name4"]].head(5)

,symbol,short_name,icb_level2,icb_name2,icb_level4,icb_name4
0,AAA,An Phát Bioplastics,1300,Hóa chất,1353,"Nhựa, cao su & sợi"
1,AAM,Thủy sản Mekong,3500,Thực phẩm và đồ uống,3573,Nuôi trồng nông & hải sản
2,AAN,Lương thực A An,3500,Thực phẩm và đồ uống,3573,Nuôi trồng nông & hải sản
3,AAT,Tập đoàn Tiên Sơn Thanh Hóa,3700,Hàng cá nhân & Gia dụng,3763,Hàng May mặc
4,ABR,Đầu tư Nhãn Hiệu Việt,2700,Hàng & Dịch vụ Công nghiệp,2791,Tư vấn & Hỗ trợ KD


Và tham số `icb=` của `meta.symbols()` nhận **cả mã cấp 2 lẫn cấp 4** — bạn
không cần biết mã mình đang cầm thuộc cấp nào:

In [4]:
print(f"icb='8300' (cấp 2, Ngân hàng): {len(client.meta.symbols(icb='8300'))} mã")
print(f"icb='8355' (cấp 4, Ngân hàng): {len(client.meta.symbols(icb='8355'))} mã")
print(f"icb='8600' (cấp 2, Bất động sản): {len(client.meta.symbols(icb='8600'))} mã")

icb='8300' (cấp 2, Ngân hàng): 28 mã
icb='8355' (cấp 4, Ngân hàng): 28 mã


icb='8600' (cấp 2, Bất động sản): 132 mã


## 2 · Chỉ số ngành — 19 ngành trong một request

`client.eod.sector.ohlcv()` trả về **chỉ số ngành** đã tính sẵn, không phải giá
bình quân của các mã trong ngành. Đơn vị là `index_point`, và cột `n_stocks`
cho biết ngành đó gồm bao nhiêu mã.

In [5]:
nganh = client.eod.sector.ohlcv(
    cap2["icb"].tolist(),
    icb_level=2,
    start=lui_ngay(HOM_NAY, nam=1),
)

print(f"{nganh['icb'].nunique()} ngành · {len(nganh):,} dòng · đơn vị {nganh.attrs['finlens']['units']['close']}")
nganh.groupby("icb_name", observed=True)["n_stocks"].last().sort_values(ascending=False).head(6)

19 ngành · 4,750 dòng · đơn vị index_point


icb_name
Xây dựng và Vật liệu             303
Hàng & Dịch vụ Công nghiệp       249
Thực phẩm và đồ uống             144
Điện, nước & xăng dầu khí đốt    140
Bất động sản                     123
Tài nguyên Cơ bản                103
Name: n_stocks, dtype: Int32

### Xếp hạng sức mạnh ngành, 12 tháng

Cùng phép chuẩn hoá gốc 100 của notebook `11` — 19 ngành khác thang điểm nhau
hoàn toàn, nên chỉ có phần trăm mới so được.

In [6]:
nganh = nganh.sort_values(["icb", "date"])
goc = nganh.groupby("icb", observed=True)["close"].transform("first")
nganh = nganh.assign(chi_so_100=nganh["close"] / goc * 100)

suc_manh = (
    nganh.groupby("icb_name", observed=True)["chi_so_100"]
    .last()
    .sub(100)
    .round(2)
    .reset_index()
    .rename(columns={"chi_so_100": "thay_doi_pct"})
)

bar_ngang(
    suc_manh,
    nhan="icb_name",
    gia_tri="thay_doi_pct",
    tieu_de="Sức mạnh ngành ICB cấp 2 — 12 tháng",
    phu_de="Chỉ số ngành chuẩn hoá về 100 tại phiên đầu kỳ",
    nhan_x="% thay đổi",
    dinh_dang_nhan="{:+.1f}%",
)

### Heatmap theo tháng: ngành nào mạnh *vào lúc nào*

Bảng xếp hạng ở trên nén cả năm vào một con số, nên nó giấu mất thời điểm. Một
ngành +30% có thể tăng đều mười hai tháng, hoặc đứng yên mười một tháng rồi
nhảy một phát — hai câu chuyện khác hẳn nhau.

In [7]:
thang = (
    nganh.assign(ky=nganh["date"].dt.to_period("M"))
    .groupby(["icb_name", "ky"], observed=True)["close"]
    .last()
    .reset_index()
)
thang["ls_thang"] = thang.groupby("icb_name", observed=True)["close"].pct_change() * 100

bang = (
    thang.dropna(subset=["ls_thang"])
    .pivot(index="icb_name", columns="ky", values="ls_thang")
    .round(1)
)
bang.columns = [str(c) for c in bang.columns]
bang = bang.loc[bang.mean(axis=1).sort_values(ascending=False).index]

heatmap(
    bang,
    tieu_de="Lợi suất theo tháng của từng ngành ICB",
    phu_de="Xếp theo lợi suất bình quân · số in trong ô vì đây là chỗ màu mang toàn bộ thông tin",
    nhan_mau="% / tháng",
)

Ô xám luôn nghĩa là "không đổi" — điểm giữa của thang màu khoá cứng ở 0 chứ
không trôi theo biên độ của bảng. Nếu để nó trôi, cùng một sắc xám sẽ mang
nghĩa khác nhau ở hai biểu đồ và không ai phát hiện ra.

## 3 · Độ rộng thị trường

Đây là phần trả lời câu hỏi mở đầu. Ba thước đo, tính trên toàn bộ cổ phiếu
HOSE:

| Thước đo | Nói lên điều gì |
|---|---|
| % số mã trên MA50 | xu hướng trung hạn có lan rộng không |
| Số mã tăng − số mã giảm | áp lực trong từng phiên |
| Đường A/D tích luỹ | dòng chảy cộng dồn của hai con số trên |

In [8]:
danh_sach = ma_hose["symbol"].tolist()

toan_san = client.eod.stock.ohlcv(danh_sach, start=lui_ngay(HOM_NAY, nam=1))
print(f"{toan_san['symbol'].nunique()} mã · {len(toan_san):,} dòng")

405 mã · 100,193 dòng


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


### % số mã trên MA50

⚠️ Đây là chỗ **bắt buộc** dùng `df.finlens.sma(50)` chứ không phải
`talib.SMA(df["close"])`. Frame này có 400 mã xếp chồng lên nhau; TA-Lib chạy
thẳng trên cột `close` sẽ lấy 49 giá cuối của mã trước để tính cửa sổ đầu của
mã sau. Notebook `31` đo con số sai cụ thể.

In [9]:
co_ma = toan_san.sort_values(["symbol", "date"]).finlens.sma(50)

do_rong = (
    co_ma.dropna(subset=["sma_50"])
    .assign(tren_ma=lambda d: (d["close"] > d["sma_50"]).astype(int))
    .groupby("date", observed=True)["tren_ma"]
    .agg(["mean", "count"])
    .reset_index()
)
do_rong["pct_tren_ma50"] = do_rong["mean"] * 100

print(f"Phiên gần nhất: {do_rong['pct_tren_ma50'].iloc[-1]:.1f}% số mã nằm trên MA50")
print(f"Bình quân 12 tháng: {do_rong['pct_tren_ma50'].mean():.1f}%")

fig = duong(
    do_rong,
    x="date",
    y="pct_tren_ma50",
    tieu_de="Độ rộng thị trường — % cổ phiếu HOSE nằm trên MA50",
    phu_de="Trên 70% là thị trường lan rộng · dưới 30% là bán tháo diện rộng",
    nhan_y="% số mã",
)
fig.add_hline(y=70, line_dash="dot", line_width=1, line_color="#898781")
fig.add_hline(y=30, line_dash="dot", line_width=1, line_color="#898781")
fig.add_hline(y=50, line_width=1, line_color="#c3c2b7")
fig

Phiên gần nhất: 34.7% số mã nằm trên MA50
Bình quân 12 tháng: 33.6%


### Phân kỳ: chỉ số đi lên trong khi độ rộng đi xuống

Đây là lý do người ta đo breadth. Đặt VNINDEX cạnh độ rộng — **hai biểu đồ
chồng dọc, không phải hai trục y**. Trục thời gian chung ở dưới; hai thang giá
trị giữ nguyên ý nghĩa của mình.

In [10]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

vni = client.eod.index.ohlcv("VNINDEX", start=lui_ngay(HOM_NAY, nam=1))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07, row_heights=[0.55, 0.45])
fig.add_trace(
    go.Scatter(x=vni["date"], y=vni["close"], name="VNINDEX", line=dict(width=2, color="#2a78d6")),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=do_rong["date"],
        y=do_rong["pct_tren_ma50"],
        name="% mã trên MA50",
        line=dict(width=2, color="#eb6834"),
    ),
    row=2,
    col=1,
)
fig.add_hline(y=50, line_width=1, line_color="#c3c2b7", row=2, col=1)
fig.update_yaxes(title_text="điểm chỉ số", row=1, col=1)
fig.update_yaxes(title_text="% số mã", row=2, col=1)
fig.update_layout(
    title_text="VNINDEX và độ rộng thị trường<br><sub style='color:#52514e'>Chỉ số lên mà độ rộng xuống = đà tăng đang thu hẹp vào ít mã</sub>",
    height=640,
    hovermode="x unified",
    showlegend=True,
)
fig

### Số mã tăng trừ số mã giảm, và đường A/D tích luỹ

In [11]:
tang_giam = toan_san.sort_values(["symbol", "date"]).assign(
    thay_doi=lambda d: d.groupby("symbol", observed=True)["close"].diff()
)

ad = (
    tang_giam.dropna(subset=["thay_doi"])
    .assign(
        tang=lambda d: (d["thay_doi"] > 0).astype(int),
        giam=lambda d: (d["thay_doi"] < 0).astype(int),
    )
    .groupby("date", observed=True)[["tang", "giam"]]
    .sum()
    .reset_index()
)
ad["thuan"] = ad["tang"] - ad["giam"]
ad["ad_tich_luy"] = ad["thuan"].cumsum()

print(f"Phiên gần nhất: {ad['tang'].iloc[-1]} mã tăng · {ad['giam'].iloc[-1]} mã giảm")

duong(
    ad,
    x="date",
    y="ad_tich_luy",
    tieu_de="Đường Advance/Decline tích luỹ — HOSE",
    phu_de="Cộng dồn (số mã tăng − số mã giảm) qua từng phiên",
    nhan_y="tích luỹ",
    moc_khong=True,
)

Phiên gần nhất: 156 mã tăng · 152 mã giảm


## 4 · Ngành nào đang dẫn dắt *tuần này*

Ghép lại: xếp hạng ngành trên ba khung thời gian cùng lúc. Một ngành mạnh ở cả
ba cột là xu hướng; mạnh ở cột tuần nhưng yếu ở cột năm là hồi phục kỹ thuật.

In [12]:
moc_cuoi = nganh["date"].max()
khung = {"1 tuần": 5, "1 tháng": 21, "3 tháng": 63}

bang_khung = {}
for ten, so_phien in khung.items():
    gan_day = nganh[nganh["date"] > moc_cuoi - pd.Timedelta(days=so_phien * 1.5)]
    dau = gan_day.groupby("icb_name", observed=True)["close"].first()
    cuoi = gan_day.groupby("icb_name", observed=True)["close"].last()
    bang_khung[ten] = ((cuoi / dau - 1) * 100).round(1)

xep_hang = pd.DataFrame(bang_khung).sort_values("1 tháng", ascending=False)

heatmap(
    xep_hang,
    tieu_de="Sức mạnh ngành trên ba khung thời gian",
    phu_de="Mạnh cả ba cột = xu hướng · chỉ mạnh cột trái = hồi kỹ thuật",
    nhan_mau="%",
)

## Tổng kết

| Bạn cần | Gọi |
|---|---|
| 19 ngành cấp 2 | `meta.sectors(level=2)` |
| Mọi mã trong một ngành | `meta.symbols(icb="8300")` — nhận cả cấp 2 lẫn cấp 4 |
| Chỉ số 19 ngành | `eod.sector.ohlcv(danh_sach_icb, icb_level=2)` |
| Ngành của từng mã | cột `icb_level2` / `icb_level4` sẵn trong `meta.symbols()` |

**Ba điều mang sang notebook sau:**

1. Cấp 2 để nhìn tổng thể, **cấp 4 để so peer** — notebook `23` chuẩn hoá định
   giá trong từng ngành cấp 4 chính vì lý do này.
2. Chỉ số tăng mà độ rộng giảm là đà tăng đang thu hẹp. Một con số chỉ số không
   nói được điều đó.
3. Heatmap phải khoá điểm giữa ở 0. Để nó trôi theo biên độ thì cùng một sắc
   xám mang nghĩa khác nhau ở hai biểu đồ.

---

**Tiếp theo:** [`13_dong_tien_nha_dau_tu.ipynb`](13_dong_tien_nha_dau_tu.ipynb)
— khối ngoại, tự doanh, và cái bẫy cộng bốn nhóm chi tiết.